In [1]:
import pandas as pd
import numpy as np

# ── 1. LOAD ──────────────────────────────────────────────────────────────────
# Try both common separators; proteomics files are sometimes tab-separated
try:
    df = pd.read_csv("data/gene expression/4_Harmonized_MS_CCLE_Gygi_subsetted.csv")  # adjust filename as needed
    if df.shape[1] == 1:
        raise ValueError("Likely tab-separated")
except:
    df = pd.read_csv("data/gene expression/4_Harmonized_MS_CCLE_Gygi_subsetted.csv", sep="\t")

print("=== SHAPE ===")
print(df.shape)

# ── 2. COLUMNS & DTYPES ──────────────────────────────────────────────────────
print("\n=== COLUMNS & DTYPES ===")
print(df.dtypes.to_string())

# ── 3. FIRST FEW ROWS ────────────────────────────────────────────────────────
print("\n=== HEAD (3) ===")
print(df.head(3).to_string())

=== SHAPE ===
(375, 12559)

=== COLUMNS & DTYPES ===
Unnamed: 0                  object
A0AV96 (RBM47)             float64
A0AVF1 (IFT56)             float64
A0AVG3 (TSNARE1)           float64
A0AVI4 (TMEM129)           float64
A0AVK6 (E2F8)              float64
A0AVT1 (UBA6)              float64
A0JLT2 (MED19)             float64
A0JNW5 (BLTP3B)            float64
A0MZ66 (SHTN1)             float64
A0PK00 (TMEM120B)          float64
A1A4S6 (ARHGAP10)          float64
A1A5B4 (ANO9)              float64
A1A5C7 (SLC22A23)          float64
A1L0T0 (ILVBL)             float64
A1L390 (PLEKHG3)           float64
A1L3X0 (ELOVL7)            float64
A1X283 (SH3PXD2B)          float64
A1XBS5 (CIBAR1)            float64
A1Z1Q3 (MACROD2)           float64
A2A288 (ZC3H12D)           float64
A2A2Y4 (FRMD3)             float64
A2RRP1 (NBAS)              float64
A2RTX5 (TARS3)             float64
A2RU49 (HYKK)              float64
A2RUB1 (MEIOC)             float64
A2RUB6 (CCDC66)            float64
A2

In [2]:
# ── 4. NULLS ─────────────────────────────────────────────────────────────────
print("\n=== NULL COUNTS (top 20 cols) ===")
nulls = df.isnull().sum()
print(nulls[nulls > 0].head(20))
print(f"Total null cells: {df.isnull().sum().sum():,}")
print(f"% missing overall: {df.isnull().mean().mean()*100:.1f}%")



=== NULL COUNTS (top 20 cols) ===
A0AVF1 (IFT56)       108
A0AVG3 (TSNARE1)     312
A0AVI4 (TMEM129)     294
A0AVK6 (E2F8)        303
A0JLT2 (MED19)        62
A0JNW5 (BLTP3B)       18
A0PK00 (TMEM120B)    143
A1A4S6 (ARHGAP10)      9
A1A5B4 (ANO9)        321
A1A5C7 (SLC22A23)    249
A1L3X0 (ELOVL7)      258
A1XBS5 (CIBAR1)      231
A1Z1Q3 (MACROD2)     321
A2A288 (ZC3H12D)     250
A2A2Y4 (FRMD3)       321
A2RTX5 (TARS3)        27
A2RU49 (HYKK)        357
A2RUB1 (MEIOC)       321
A2RUB6 (CCDC66)      330
A2RUC4 (TYW5)         53
dtype: int64
Total null cells: 1,308,704
% missing overall: 27.8%


In [3]:
# ── 5. ORIENTATION ───────────────────────────────────────────────────────────
# Is it wide (cell lines as columns) or long (one row per gene×cell)?
print("\n=== FORMAT GUESS ===")
print(f"Rows: {df.shape[0]:,}  |  Cols: {df.shape[1]:,}")
if df.shape[1] > df.shape[0]:
    print("→ Likely WIDE format (genes as rows, cell lines as columns)")
else:
    print("→ Likely LONG format (one row per measurement)")


=== FORMAT GUESS ===
Rows: 375  |  Cols: 12,559
→ Likely WIDE format (genes as rows, cell lines as columns)


In [4]:
# ── 6. INDEX / ID COLUMNS ────────────────────────────────────────────────────
print("\n=== FIRST COLUMN SAMPLE (potential gene/protein ID) ===")
print(df.iloc[:10, 0])


=== FIRST COLUMN SAMPLE (potential gene/protein ID) ===
0    ACH-000849
1    ACH-000441
2    ACH-000248
3    ACH-000684
4    ACH-000856
5    ACH-000348
6    ACH-000062
7    ACH-000650
8    ACH-000484
9    ACH-000625
Name: Unnamed: 0, dtype: object


In [5]:
# ── 7. GENE IDENTIFIER TYPE ──────────────────────────────────────────────────
id_col = df.columns[0]
sample_ids = df[id_col].dropna().astype(str).head(20)
print("\n=== ID COLUMN SAMPLE ===")
print(sample_ids.tolist())

has_ensg   = sample_ids.str.match(r'^ENSG\d+').any()
has_symbol = sample_ids.str.match(r'^[A-Z][A-Z0-9]+$').any()
has_uniprot = sample_ids.str.match(r'^[OPQ][0-9][A-Z0-9]{3}[0-9]').any()
print(f"Contains ENSG IDs:    {has_ensg}")
print(f"Contains gene symbols:{has_symbol}")
print(f"Contains UniProt IDs: {has_uniprot}")


=== ID COLUMN SAMPLE ===
['ACH-000849', 'ACH-000441', 'ACH-000248', 'ACH-000684', 'ACH-000856', 'ACH-000348', 'ACH-000062', 'ACH-000650', 'ACH-000484', 'ACH-000625', 'ACH-000739', 'ACH-000361', 'ACH-001239', 'ACH-000250', 'ACH-000014', 'ACH-000713', 'ACH-000753', 'ACH-000023', 'ACH-000711', 'ACH-000278']
Contains ENSG IDs:    False
Contains gene symbols:False
Contains UniProt IDs: False


In [6]:

# ── 8. CELL LINE IDs IN COLUMNS (if wide) ────────────────────────────────────
if df.shape[1] > 100:
    cols = pd.Series(df.columns[1:])
    print("\n=== COLUMN ID SAMPLE (cell lines?) ===")
    print(cols.head(10).tolist())
    has_ach  = cols.str.startswith('ACH-').any()
    has_cvcl = cols.str.startswith('CVCL').any()
    has_pr   = cols.str.startswith('PR-').any()
    print(f"ACH- IDs: {has_ach}  |  CVCL: {has_cvcl}  |  PR-: {has_pr}")


=== COLUMN ID SAMPLE (cell lines?) ===
['A0AV96 (RBM47)', 'A0AVF1 (IFT56)', 'A0AVG3 (TSNARE1)', 'A0AVI4 (TMEM129)', 'A0AVK6 (E2F8)', 'A0AVT1 (UBA6)', 'A0JLT2 (MED19)', 'A0JNW5 (BLTP3B)', 'A0MZ66 (SHTN1)', 'A0PK00 (TMEM120B)']
ACH- IDs: False  |  CVCL: False  |  PR-: False


In [7]:

# ── 9. VALUE DISTRIBUTION (numeric columns) ──────────────────────────────────
numeric_cols = df.select_dtypes(include=np.number).columns
if len(numeric_cols) > 0:
    print("\n=== NUMERIC VALUE DISTRIBUTION (sample of cols) ===")
    sample_num = df[numeric_cols[:50]]
    vals = sample_num.values.flatten()
    vals = vals[~np.isnan(vals)]
    print(f"Min: {vals.min():.4f}  |  Max: {vals.max():.4f}  |  Mean: {vals.mean():.4f}  |  Median: {np.median(vals):.4f}")
    print(f"% zeros: {(vals == 0).mean()*100:.1f}%")
    print(f"% negative: {(vals < 0).mean()*100:.1f}%")
    # Are values already log-transformed? Log values are typically -5 to 20
    if vals.max() < 50:
        print("→ Values likely already log-transformed (range < 50)")
    else:
        print("→ Values likely raw (range > 50) — may need log transform")


=== NUMERIC VALUE DISTRIBUTION (sample of cols) ===
Min: -16.3773  |  Max: 7.0426  |  Mean: -0.0271  |  Median: -0.0402
% zeros: 0.0%
% negative: 52.3%
→ Values likely already log-transformed (range < 50)


In [8]:

# ── 10. UNIQUE GENES / PROTEINS ──────────────────────────────────────────────
print(f"\n=== UNIQUE VALUES IN ID COLUMN ===")
print(f"Unique: {df[id_col].nunique():,}")
print(f"Duplicates in ID col: {df[id_col].duplicated().sum()}")


=== UNIQUE VALUES IN ID COLUMN ===
Unique: 375
Duplicates in ID col: 0


In [9]:

# ── 11. COVERAGE HEATMAP (missingness per column) ────────────────────────────
if df.shape[1] > 5:
    miss_per_col = df[numeric_cols].isnull().mean().sort_values(ascending=False)
    print("\n=== TOP 10 COLS BY MISSINGNESS ===")
    print(miss_per_col.head(10))
    print(f"\nCols with >50% missing: {(miss_per_col > 0.5).sum()}")
    print(f"Cols with 100% missing: {(miss_per_col == 1.0).sum()}")


=== TOP 10 COLS BY MISSINGNESS ===
P17752-2 (TPH1)       0.978667
P01258-2 (CALCA)      0.978667
Q13291 (SLAMF1)       0.978667
P07288 (KLK3)         0.978667
Q6MZQ0 (PRR5L)        0.978667
P63135 (ERVK-7)       0.978667
Q14524 (SCN5A)        0.978667
P81408-4 (ENTREP3)    0.978667
Q03403 (TFF2)         0.976000
Q8TC41 (RNF217)       0.976000
dtype: float64

Cols with >50% missing: 3544
Cols with 100% missing: 0


In [10]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("data/gene expression/4_Harmonized_MS_CCLE_Gygi_subsetted.csv", index_col=0)

# ── 1. CONFIRM VALUE TYPE ────────────────────────────────────────────────────
# Is this log2 intensity, log2 ratio, or z-score?
# Check a well-known protein like EGFR (P00533) distribution
egfr_col = [c for c in df.columns if 'EGFR' in c or 'P00533' in c]
print("=== EGFR column(s) ===")
print(egfr_col)
if egfr_col:
    print(df[egfr_col[0]].describe())

=== EGFR column(s) ===
['P00533 (EGFR)']
count    375.000000
mean      -0.008865
std        1.579554
min       -3.178944
25%       -1.238981
50%       -0.005592
75%        1.011990
max        5.154394
Name: P00533 (EGFR), dtype: float64


In [11]:


# ── 2. ISOFORM HANDLING ──────────────────────────────────────────────────────
# How many isoform entries vs canonical?
canonical = [c for c in df.columns if not re.search(r'-\d+\s*\(', c) and not re.search(r'-\d+$', c)]
isoforms  = [c for c in df.columns if re.search(r'-\d+[\s(]', c) or re.search(r'-\d+$', c)]
print(f"\n=== ISOFORMS ===")
print(f"Canonical proteins: {len(canonical):,}")
print(f"Isoform entries:    {len(isoforms):,}")
print(f"Sample isoforms: {isoforms[:5]}")


=== ISOFORMS ===
Canonical proteins: 10,916
Isoform entries:    1,642
Sample isoforms: ['O15079-2 (SNPH)', 'A0FGR8-2 (ESYT2)', 'O00469-2 (PLOD2)', 'O15054-1 (KDM6B)', 'A8K8P3-3 (SFI1)']


In [12]:

# ── 3. COVERAGE PER CELL LINE (row) ─────────────────────────────────────────
# Are some cell lines data-sparse?
coverage = df.notna().sum(axis=1)
print(f"\n=== COVERAGE PER CELL LINE ===")
print(f"Min proteins detected: {coverage.min()}")
print(f"Max proteins detected: {coverage.max()}")
print(f"Mean proteins detected: {coverage.mean():.0f}")
print(f"Cell lines with <5000 proteins: {(coverage < 5000).sum()}")
print(f"Cell lines with <2000 proteins: {(coverage < 2000).sum()}")


=== COVERAGE PER CELL LINE ===
Min proteins detected: 8096
Max proteins detected: 10425
Mean proteins detected: 9068
Cell lines with <5000 proteins: 0
Cell lines with <2000 proteins: 0


In [13]:
# ── 4. EXTRACT GENE SYMBOLS FROM COLUMN NAMES ───────────────────────────────
# Parse: "P00533 (EGFR)" → gene symbol = "EGFR"
symbols = {}
for col in df.columns:
    m = re.search(r'\(([^)]+)\)', col)
    if m:
        symbols[col] = m.group(1)
    else:
        symbols[col] = None  # bare UniProt with no symbol

no_symbol = [c for c in df.columns if symbols[c] is None]
print(f"\n=== GENE SYMBOL EXTRACTION ===")
print(f"Columns with no symbol: {len(no_symbol)}")
print(f"Sample no-symbol cols: {no_symbol[:10]}")

# ── 5. DUPLICATE GENE SYMBOLS (multiple UniProt entries per gene) ────────────
from collections import Counter
sym_counts = Counter(s for s in symbols.values() if s)
multi_uniprot = {g: n for g, n in sym_counts.items() if n > 1}
print(f"\n=== DUPLICATE GENE SYMBOLS ===")
print(f"Genes with >1 UniProt column: {len(multi_uniprot):,}")
print(f"Top 10 most duplicated: {sorted(multi_uniprot.items(), key=lambda x: -x[1])[:10]}")


=== GENE SYMBOL EXTRACTION ===
Columns with no symbol: 17
Sample no-symbol cols: ['O00370', 'E9PSI1', 'B4E1Z4', 'B7Z1Y9', 'F8W031', 'H0YHG0', 'H7C469', 'B4DKC2', 'B4DN88', 'B4DXA9']

=== DUPLICATE GENE SYMBOLS ===
Genes with >1 UniProt column: 396
Top 10 most duplicated: [('PLEC', 6), ('MAP4', 5), ('SEC16A', 4), ('TPM1', 4), ('MACF1', 4), ('PML', 4), ('BAIAP2', 3), ('ZC3HAV1', 3), ('SLC3A2', 3), ('TJP1', 3)]


In [14]:

# ── 6. GROUND TRUTH SANITY CHECK ─────────────────────────────────────────────
# Check EGFR in A431, ERBB2 in SKBR3
gt = {'EGFR': 'A431', 'ERBB2': 'SKBR3'}
print(f"\n=== GROUND TRUTH SANITY CHECK ===")
for gene, cell in gt.items():
    gene_cols = [c for c in df.columns if f'({gene})' in c and '-' not in c.split('(')[0].strip()[-3:]]
    ach_rows  = [i for i in df.index if cell.upper() in i.upper()]
    print(f"{gene} cols: {gene_cols[:3]}")
    # Need to match ACH- to cell line name — just show raw values across all rows
    if gene_cols:
        vals = df[gene_cols[0]].dropna()
        print(f"  {gene_cols[0]}: {len(vals)} non-null, mean={vals.mean():.3f}, max={vals.max():.3f}")


=== GROUND TRUTH SANITY CHECK ===
EGFR cols: ['P00533 (EGFR)']
  P00533 (EGFR): 375 non-null, mean=-0.009, max=5.154
ERBB2 cols: ['P04626 (ERBB2)']
  P04626 (ERBB2): 375 non-null, mean=0.020, max=5.147


In [15]:


# ── 7. VALUE RANGE: IS THIS Z-SCORED OR RAW LOG2? ───────────────────────────
# Z-scored data should have mean~0 and std~1 per column
col_means = df.select_dtypes(include=np.number).mean()
col_stds  = df.select_dtypes(include=np.number).std()
print(f"\n=== COLUMN-WISE STATS (per protein across cell lines) ===")
print(f"Mean of col means: {col_means.mean():.4f}  (z-scored → ~0)")
print(f"Mean of col stds:  {col_stds.mean():.4f}   (z-scored → ~1)")
print(f"Std of col means:  {col_means.std():.4f}")


=== COLUMN-WISE STATS (per protein across cell lines) ===
Mean of col means: -0.0368  (z-scored → ~0)
Mean of col stds:  0.9923   (z-scored → ~1)
Std of col means:  0.1258


In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("data/gene expression/4_Harmonized_MS_CCLE_Gygi_subsetted.csv", index_col=0)

# ── 1. CANONICAL-ONLY FILTER ─────────────────────────────────────────────────
# Keep only columns without isoform suffix (no -N before space or end)
is_isoform = df.columns.str.contains(r'-\d+[\s(]|-\d+$', regex=True)
df_canonical = df.loc[:, ~is_isoform]
print(f"Canonical columns kept: {df_canonical.shape[1]:,}  (dropped {is_isoform.sum():,} isoforms)")

In [ ]:

# ── 2. BUILD UNIPROT → GENE SYMBOL MAP ──────────────────────────────────────
def parse_col(col):
    m = re.match(r'^(\S+)\s*(?:\(([^)]+)\))?$', col.strip())
    if m:
        return m.group(1), m.group(2)  # (uniprot, symbol)
    return col, None

col_map = {col: parse_col(col) for col in df_canonical.columns}
# Columns with no symbol
no_sym = [c for c, (u, s) in col_map.items() if s is None]
print(f"No-symbol canonical cols: {len(no_sym)}")

In [ ]:



# ── 3. HANDLE DUPLICATE GENE SYMBOLS — collapse to max per gene ──────────────
# Build a gene→[cols] lookup
from collections import defaultdict
gene_to_cols = defaultdict(list)
for col, (uniprot, sym) in col_map.items():
    key = sym if sym else uniprot
    gene_to_cols[key].append(col)

multi = {g: cols for g, cols in gene_to_cols.items() if len(cols) > 1}
print(f"\nGenes with multiple UniProt cols: {len(multi):,}")
print("Sample:", {k: v for k, v in list(multi.items())[:3]})

# Collapse: for each gene with duplicates, take max across its UniProt columns
# (max is appropriate: highest detected isoform = gene is present)
collapsed_extras = {}
for gene, cols in multi.items():
    if len(cols) > 1:
        collapsed_extras[gene] = df_canonical[cols].max(axis=1)

# Rebuild: start with single-entry genes, then overwrite multi-entry genes
single_cols = [col for col, (u, s) in col_map.items()
               if (s if s else u) not in multi]
df_single = df_canonical[single_cols].copy()
# Rename to gene symbol
df_single.columns = [col_map[c][1] if col_map[c][1] else col_map[c][0]
                     for c in df_single.columns]

df_multi = pd.DataFrame(collapsed_extras, index=df_canonical.index)

df_collapsed = pd.concat([df_single, df_multi], axis=1)
print(f"\nShape after collapse: {df_collapsed.shape}")
print(f"Expected ~{len(gene_to_cols):,} unique genes")

# ── 4. MELT TO LONG FORMAT (for KG ingestion) ────────────────────────────────
df_long = df_collapsed.reset_index().melt(
    id_vars='Unnamed: 0',
    var_name='gene_symbol',
    value_name='protein_zscore'
).rename(columns={'Unnamed: 0': 'ACH_ID'})

df_long = df_long.dropna(subset=['protein_zscore'])

print(f"\n=== LONG FORMAT ===")
print(f"Shape: {df_long.shape}")
print(f"Unique cell lines: {df_long['ACH_ID'].nunique()}")
print(f"Unique genes:      {df_long['gene_symbol'].nunique()}")
print(f"Null protein scores remaining: {df_long['protein_zscore'].isna().sum()}")
print(f"\nSample rows:")
print(df_long.head(5).to_string(index=False))

# ── 5. GROUND TRUTH VALIDATION ───────────────────────────────────────────────
# Load sample info to get cell line names
sample_info = pd.read_csv("9_DepMap_sample_info.csv")
ach_to_name = sample_info.set_index('DepMap_ID')['stripped_cell_line_name'].to_dict()

df_long['cell_line_name'] = df_long['ACH_ID'].map(ach_to_name)

gt_checks = [('EGFR', 'A431'), ('ERBB2', 'SKBR3')]
print(f"\n=== GROUND TRUTH PROTEIN LEVELS ===")
for gene, cell in gt_checks:
    rows = df_long[(df_long['gene_symbol'] == gene) &
                   (df_long['cell_line_name'].str.upper() == cell.upper())]
    if not rows.empty:
        score = rows['protein_zscore'].values[0]
        # Rank among all cell lines for this gene
        all_vals = df_long[df_long['gene_symbol'] == gene]['protein_zscore']
        rank = (all_vals > score).sum() + 1
        pct = round(rank / len(all_vals) * 100, 1)
        print(f"  {gene} in {cell}: z={score:.3f}, rank {rank}/{len(all_vals)} (top {pct}%)")
    else:
        print(f"  {gene} in {cell}: NOT FOUND")

# ── 6. SUMMARY STATS ─────────────────────────────────────────────────────────
print(f"\n=== FINAL SUMMARY ===")
print(f"Cell lines:     {df_long['ACH_ID'].nunique()}")
print(f"Proteins:       {df_long['gene_symbol'].nunique()}")
print(f"Measurements:   {len(df_long):,}")
print(f"Missing flagged as below-detection (dropped from long): "
      f"{df_collapsed.isna().sum().sum():,}")

Canonical columns kept: 10,916  (dropped 1,642 isoforms)
No-symbol canonical cols: 17

Genes with multiple UniProt cols: 110
Sample: {'SDCBP': ['O00560 (SDCBP)', 'G5EA09 (SDCBP)'], 'SYNM': ['O15061 (SYNM)', 'C9JIE4 (SYNM)'], 'HMGB3': ['O15347 (HMGB3)', 'E7ES08 (HMGB3)']}

Shape after collapse: (375, 10803)
Expected ~10,803 unique genes


KeyError: "The following id_vars or value_vars are not present in the DataFrame: ['Unnamed: 0']"